# Taxi fares with the Huber loss

A taxi company wants its fare rule, a flag fee plus a rate per kilometre, back from 45
recorded trips. On three of the trips the driver forgot to stop the meter at the drop-off,
and those fares came out far too high. All three errors point the same way, so least
squares tilts the whole line toward them. You will fit the line under the **Huber loss**
instead, which is squared for small residuals and linear for large ones:

$$
\rho_\delta(r) = \begin{cases} \tfrac12 r^2 & |r| \le \delta \\ \delta\,(|r| - \tfrac12\delta) & |r| > \delta \end{cases}
$$

The fit uses **iteratively reweighted least squares**: fit, give each row a weight that
shrinks with the size of its residual, fit again with those weights, and repeat.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 45 trips: x is the distance in km, y the fare in dollars.
rng = np.random.default_rng(23)
n = 45
x = np.round(rng.uniform(1.0, 18.0, n), 1)
y = np.round(3.0 + 1.8 * x + rng.normal(0.0, 0.6, n), 2)
runaway = [9, 26, 38]                             # the meter kept running after drop-off
y[runaway] += np.array([21.0, 34.0, 27.0])
X = np.column_stack([np.ones(n), x])              # flag fee and per-km columns
ols = np.linalg.lstsq(X, y, rcond=None)[0]
print(f"least squares: intercept {ols[0]:.3f}, slope {ols[1]:.3f}")

## Part 1 (coding task): The Huber loss and its weights

Write two functions of an array of residuals `r` and a threshold `delta > 0`.

- `huber(r, delta)` returns $\rho_\delta(r)$ for every entry.
- `huber_weight(r, delta)` returns $\psi(r) / r$ for every entry, where
  $\psi = \rho_\delta'$ is the derivative of the loss. Work out $\psi$ on each side of
  $\delta$. At $r = 0$ the weight is 1.

The weight is what makes the reweighting work: minimizing $\sum_i w_i r_i^2 / 2$ with
$w_i = \psi(r_i)/r_i$ held fixed has the same gradient as $\sum_i \rho_\delta(r_i)$ at the
current fit.

In [ ]:
def huber(r, delta):
    r = np.asarray(r, dtype=float)
    # TODO
    raise NotImplementedError


def huber_weight(r, delta):
    r = np.asarray(r, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
r = np.array([0.0, 0.5, -1.0, 2.0, -3.0, 10.0])
assert np.allclose(huber(r, 1.0), [0.0, 0.125, 0.5, 1.5, 2.5, 9.5]), f"huber(r, 1) gave {huber(r, 1.0)}: check the 1/2 in the squared part and the -delta/2 in the linear part"
assert np.allclose(huber(r, 2.0), [0.0, 0.125, 0.5, 2.0, 4.0, 18.0]), f"huber(r, 2) gave {huber(r, 2.0)}"
eps = 1e-9
assert abs(huber(1.5 + eps, 1.5) - huber(1.5 - eps, 1.5)) < 1e-6, "the loss should be continuous at |r| = delta"
w = huber_weight(r, 1.0)
assert np.all(w > 0), f"weights must be positive, got {w}: psi(r) and r have the same sign"
assert np.allclose(w, [1.0, 1.0, 1.0, 0.5, 1 / 3, 0.1]), f"huber_weight(r, 1) gave {w}: outside delta, psi(r) = delta * sign(r), so the weight is delta / |r|"
assert np.allclose(huber_weight(r, 2.0), [1.0, 1.0, 1.0, 1.0, 2 / 3, 0.2]), "wrong weights for delta = 2"
print("Part 1 passed")

## Part 2 (coding task): Weighted least squares

Write `weighted_lstsq(X, y, w)`, the coefficients $\beta$ that minimize
$\sum_i w_i\,(y_i - \mathbf{x}_i^\top\beta)^2$ for nonnegative weights `w`. Setting the
gradient to zero gives the normal equations $X^\top W X \beta = X^\top W y$ with
$W = \operatorname{diag}(w)$. Solve them with `np.linalg.solve`, or scale each row of `X` and
`y` by $\sqrt{w_i}$ and call `np.linalg.lstsq`.

In [ ]:
def weighted_lstsq(X, y, w):
    X, y, w = np.asarray(X, dtype=float), np.asarray(y, dtype=float), np.asarray(w, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
Xs = np.column_stack([np.ones(4), [0.0, 1.0, 2.0, 3.0]])
ys = np.array([1.0, 2.0, 2.0, 7.0])
assert np.allclose(weighted_lstsq(Xs, ys, np.ones(4)), np.linalg.lstsq(Xs, ys, rcond=None)[0]), "with equal weights the answer is ordinary least squares"
assert np.allclose(weighted_lstsq(Xs, ys, [1, 1, 1, 0]), [1.1666666666666667, 0.5]), "a weight of 0 should drop the row: the first three points alone give intercept 7/6 and slope 1/2"
assert np.allclose(weighted_lstsq(Xs, ys, [1, 1, 1, 0.25]), [27 / 38, 45 / 38]), f"with weights (1, 1, 1, 1/4) expected intercept 27/38 and slope 45/38, got {weighted_lstsq(Xs, ys, [1, 1, 1, 0.25])}: weight each squared residual by w, not by w squared or the square root of w"
assert np.allclose(weighted_lstsq(Xs, ys, [3, 3, 3, 0.75]), weighted_lstsq(Xs, ys, [1, 1, 1, 0.25])), "scaling every weight by the same factor should not change the fit"
print("Part 2 passed")

## Part 3 (coding task): Iteratively reweighted least squares

Write `huber_fit(X, y, delta, iters=100)`. Start from the ordinary least-squares fit. Then
repeat: compute the residuals $r = y - X\beta$, compute the weights with `huber_weight`, and
refit with `weighted_lstsq`. Stop after `iters` rounds or as soon as $\beta$ changes by less
than `1e-12`, and return $\beta$.

The check tests that the answer minimizes the Huber objective $\sum_i \rho_\delta(r_i)$: at the
minimum its gradient, $-\sum_i \psi(r_i)\,\mathbf{x}_i$, is zero. It then fits the
taxi trips with $\delta = 1$.

In [ ]:
def huber_fit(X, y, delta, iters=100):
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def huber_gradient(X, y, beta, delta):
    r = y - X @ beta
    psi = huber_weight(r, delta) * r                 # psi(r) = weight * r
    return -X.T @ psi

beta = np.asarray(huber_fit(X, y, 1.0))
assert beta.shape == (2,), f"return the two coefficients, got shape {beta.shape}"
g = huber_gradient(X, y, beta, 1.0)
assert np.max(np.abs(g)) < 1e-6, f"the Huber objective's gradient at your fit is {g}, not zero: keep reweighting and refitting until the coefficients stop changing"
obj = lambda b: huber(y - X @ b, 1.0).sum()
assert obj(beta) < obj(ols), "the Huber fit should have a lower Huber objective than least squares"
for b in (beta + [0.05, 0.0], beta - [0.0, 0.005]):
    assert obj(beta) <= obj(b), "a nearby line has a lower Huber objective, so the fit is not the minimum"
w = huber_weight(y - X @ beta, 1.0)
assert set(np.flatnonzero(w < 0.2)) >= set(runaway), "every runaway-meter trip should end with a small weight"
print(f"least squares: intercept {ols[0]:.3f}, slope {ols[1]:.3f}")
print(f"Huber, delta 1: intercept {beta[0]:.3f}, slope {beta[1]:.3f}")
print(f"rows with weight below 1: {np.flatnonzero(w < 1).tolist()}")
assert abs(beta[1] - 1.8) < 0.05 and abs(beta[0] - 3.0) < 0.4, "the Huber line should land near the true fare rule, 3.0 plus 1.8 per km"
print("Part 3 passed")

## Solution

### Part 1: The Huber loss and its weights

In [ ]:
def huber(r, delta):
    r = np.asarray(r, dtype=float)
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r**2, delta * (a - 0.5 * delta))


def huber_weight(r, delta):
    r = np.asarray(r, dtype=float)
    a = np.abs(r)
    w = np.ones_like(r)
    outside = a > delta
    w[outside] = delta / a[outside]                  # psi(r) / r = delta / |r| out here
    return w

**Why it works.** Inside $\delta$ the loss is $r^2/2$, so $\psi(r) = r$ and the weight is 1: the row
counts as in least squares. Outside, the loss grows linearly with slope $\delta$, so
$\psi(r) = \delta\,\operatorname{sign}(r)$ and the weight is $\delta / |r|$. A row ten times
past the threshold gets a tenth of the weight, and its pull on the fit, weight times residual,
is capped at $\delta$ however wild the reading is. The $-\tfrac12\delta$ in the linear piece
makes the two pieces meet at $|r| = \delta$ with the same value and the same slope.

### Part 2: Weighted least squares

In [ ]:
def weighted_lstsq(X, y, w):
    X, y, w = np.asarray(X, dtype=float), np.asarray(y, dtype=float), np.asarray(w, dtype=float)
    XtW = X.T * w                                    # X^T W without building W
    return np.linalg.solve(XtW @ X, XtW @ y)

**Why it works.** The objective is $(y - X\beta)^\top W (y - X\beta)$, and its gradient
$-2X^\top W(y - X\beta)$ vanishes at the solution of the normal equations. `X.T * w`
multiplies column $i$ of $X^\top$ by $w_i$, which is $X^\top W$ without the $n \times n$
diagonal matrix. A weight of 0 removes a row, equal weights give ordinary least squares, and
multiplying all weights by one constant multiplies both sides of the equations by it.

### Part 3: Iteratively reweighted least squares

In [ ]:
def huber_fit(X, y, delta, iters=100):
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    for _ in range(iters):
        w = huber_weight(y - X @ beta, delta)
        new = weighted_lstsq(X, y, w)
        if np.max(np.abs(new - beta)) < 1e-12:
            return new
        beta = new
    return beta

**Why it works.** At a fixed point of the loop the weights come from the current residuals, so the
weighted normal equations read $X^\top W(\beta)(y - X\beta) = 0$, which is
$\sum_i \psi(r_i)\,\mathbf{x}_i = 0$, exactly the condition for a minimum of the Huber
objective. The objective is convex, so that minimum is the global one. Each round also lowers
the objective, because the weighted squared error is a quadratic that touches the Huber loss
at the current residuals and lies above it elsewhere. One round from least squares is not
enough, because the first weights are measured against a line the three runaway fares have
already pulled up. On the taxi trips least squares puts the rate at 2.02 dollars per km, while
the Huber fit gives 1.77, close to the true 1.8, and the three runaway trips end with weights
0.048, 0.029 and 0.037.